# Количественное сравнение методов unfold_

Этот ноутбук проводит комплексное сравнение всех методов восстановления спектра в bssunfold.
Используется база данных спектров IAEA из папки tests/ для тестирования.

## Цели:
1. Перебрать все параметры каждого метода
2. Составить таблицу эффективности восстановления спектра
3. Выдать отчет по методам с метриками
4. Построить графики спектров

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from datetime import datetime
from typing import Dict, List, Tuple, Any, Optional
import time

warnings.filterwarnings('ignore')

from bssunfold import Detector, RF_LANL

## Загрузка данных

In [ ]:
det = Detector(RF_LANL)
print(f"Количество энергетических бинов: {det.n_energy_bins}")
print(f"Количество сфер детектора: {len(det.sphere_names)}")
print(f"Сферы: {det.sphere_names}")

In [ ]:
spectra_df = pd.read_csv('../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv')
print(f"Размер базы спектров: {spectra_df.shape}")
spectrum_names = [col for col in spectra_df.columns if col != 'E_MeV']
print(f"Названия спектров: {len(spectrum_names)}")

In [ ]:
def get_reference_spectrum(df, name):
    return {'E_MeV': df['E_MeV'].values, 'Phi': df[name].values}

ref = get_reference_spectrum(spectra_df, spectrum_names[0])
readings = det.get_effective_readings_for_spectra(ref)
print(f"Показания для {spectrum_names[0]}: {list(readings.keys())}")

In [ ]:
UNFOLD_METHODS = {
    'mlem': {'func': det.unfold_mlem, 'params': [{'iterations': 10}, {'iterations': 50}, {'iterations': 100}]},
    'maxed': {'func': det.unfold_maxed, 'params': [{'sigma_factor': 0.01}, {'sigma_factor': 0.05}, {'sigma_factor': 0.1}]},
    'tsvd': {'func': det.unfold_tsvd, 'params': [{'n_components': 3}, {'n_components': 5}, {'n_components': 7}]},
    'bayes': {'func': det.unfold_bayes, 'params': [{'iterations': 100}, {'iterations': 500}]},
    'cvxpy': {'func': det.unfold_cvxpy, 'params': [{'method': 'ECOS'}, {'method': 'SCS'}]},
    'landweber': {'func': det.unfold_landweber, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'sart': {'func': det.unfold_sart, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'kaczmarz': {'func': det.unfold_kaczmarz, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'cgls': {'func': det.unfold_cgls, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'lanczos': {'func': det.unfold_lanczos, 'params': [{'n_components': 3}, {'n_components': 5}]},
    'tikhonov_tv': {'func': det.unfold_tikhonov_tv, 'params': [{'alpha': 0.001}, {'alpha': 0.01}]},
    'gravel': {'func': det.unfold_gravel, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'doroshenko': {'func': det.unfold_doroshenko, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'bunki': {'func': det.unfold_bunki, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'sandii': {'func': det.unfold_sandii, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'osem': {'func': det.unfold_osem, 'params': [{'iterations': 10}, {'iterations': 50}]},
    'fista': {'func': det.unfold_fista, 'params': [{'iterations': 10}, {'iterations': 50}]},
}

In [ ]:
KEY_METRICS = ['r2_score', 'pearson_r', 'root_mean_squared_error', 'mape', 'wasserstein_dist', 'kl_divergence', 'cosine_similarity']

In [ ]:
def evaluate_method(method_name, method_info, readings, ref_spec, det):
    results = []
    func = method_info['func']
    for params in method_info['params']:
        t0 = time.time()
        try:
            res = func(readings, **params)
            elapsed = time.time() - t0
            metrics = det.compare(ref_spec, res, metrics=KEY_METRICS)
            entry = {'method': method_name, 'params': str(params), 'success': True, 'time_sec': elapsed}
            entry.update(metrics)
        except Exception as e:
            elapsed = time.time() - t0
            entry = {'method': method_name, 'params': str(params), 'success': False, 'time_sec': elapsed, 'error': str(e)}
        results.append(entry)
    return results

In [ ]:
TEST_SPECTRA = spectrum_names[:3]
SELECTED_METHODS = list(UNFOLD_METHODS.keys())
print(f"Тестируем {len(SELECTED_METHODS)} методов на {len(TEST_SPECTRA)} спектрах")

In [ ]:
all_results = []
for spec_name in TEST_SPECTRA:
    print(f"\nСпектр: {spec_name}")
    ref_spec = get_reference_spectrum(spectra_df, spec_name)
    readings = det.get_effective_readings_for_spectra(ref_spec)
    for mname in SELECTED_METHODS:
        print(f"  {mname}...", end=' ')
        res = evaluate_method(mname, UNFOLD_METHODS[mname], readings, ref_spec, det)
        all_results.extend(res)
        ok = sum(1 for r in res if r.get('success'))
        print(f"{ok}/{len(res)} успешно")

results_df = pd.DataFrame(all_results)
print(f"\nВсего результатов: {len(results_df)}")

In [ ]:
summary = results_df.groupby('method')[KEY_METRICS].mean().reset_index()
std_dev = results_df.groupby('method')[KEY_METRICS].std().reset_index()
summary.columns = ['method'] + [f'{c}_mean' for c in KEY_METRICS]
std_dev.columns = ['method'] + [f'{c}_std' for c in KEY_METRICS]
summary_table = pd.merge(summary, std_dev, on='method')
print("Сводная таблица метрик:")
print(summary_table.to_string())

In [ ]:
ranking = summary_table.sort_values('r2_score_mean', ascending=False)
print("\nРейтинг по R²:")
print(ranking[['method', 'r2_score_mean']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_plot = [('r2_score_mean', 'R² Score', 'steelblue'),
                ('root_mean_squared_error_mean', 'RMSE', 'coral'),
                ('pearson_r_mean', 'Pearson R', 'green'),
                ('cosine_similarity_mean', 'Cosine Similarity', 'purple')]
for ax, (col, title, color) in zip(axes.flat, metrics_plot):
    data = summary_table.sort_values(col, ascending=('RMSE' in title))
    ax.barh(data['method'], data[col], color=color, alpha=0.7)
    ax.set_xlabel(title)
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('method_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
time_avg = results_df.groupby('method')['time_sec'].mean().reset_index()
time_avg.columns = ['method', 'avg_time']
merged = pd.merge(time_avg, summary, on='method')
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(merged['avg_time'], merged['r2_score_mean'], s=100, alpha=0.6)
for _, row in merged.iterrows():
    ax.annotate(row['method'], (row['avg_time'], row['r2_score_mean']), fontsize=8)
ax.set_xlabel('Время (сек)')
ax.set_ylabel('R² Score')
ax.set_title('Точность vs Время')
ax.grid(True, alpha=0.3)
plt.savefig('time_vs_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
first_spec = TEST_SPECTRA[0]
ref_spec = get_reference_spectrum(spectra_df, first_spec)
readings = det.get_effective_readings_for_spectra(ref_spec)
best_method = ranking.iloc[0]['method']
best_params = UNFOLD_METHODS[best_method]['params'][0]
best_result = UNFOLD_METHODS[best_method]['func'](readings, **best_params)
print(f"Лучший метод: {best_method}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
energy = ref_spec['E_MeV']
axes[0].plot(energy, ref_spec['Phi'], 'k-', lw=2, label='Эталон')
axes[0].plot(energy, best_result['spectrum'], 'r--', lw=2, label=f'{best_method}')
axes[0].set_xlabel('Энергия (MeV)')
axes[0].set_ylabel('Φ(E)')
axes[0].set_title(f'{first_spec} (линейный)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].semilogy(energy, ref_spec['Phi'], 'k-', lw=2, label='Эталон')
axes[1].semilogy(energy, best_result['spectrum'], 'r--', lw=2, label=f'{best_method}')
axes[1].set_xlabel('Энергия (MeV)')
axes[1].set_ylabel('Φ(E) [log]')
axes[1].set_title(f'{first_spec} (лог)')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'spectrum_{first_spec}.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.semilogy(energy, ref_spec['Phi'], 'k-', lw=3, label='Эталон', zorder=10)
colors = plt.cm.tab10(np.linspace(0, 1, min(5, len(SELECTED_METHODS))))
for i, mname in enumerate(SELECTED_METHODS[:5]):
    try:
        res = UNFOLD_METHODS[mname]['func'](readings, **UNFOLD_METHODS[mname]['params'][0])
        ax.semilogy(energy, res['spectrum'], '--', lw=2, color=colors[i], label=mname, alpha=0.8)
    except: pass
ax.set_xlabel('Энергия (MeV)')
ax.set_ylabel('Φ(E) [log]')
ax.set_title(f'Сравнение методов: {first_spec}')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('multi_method.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
try:
    import seaborn as sns
    from sklearn.preprocessing import MinMaxScaler
    hm_data = summary.copy()
    hm_data.index = hm_data['method']
    hm_cols = [c for c in hm_data.columns if c.endswith('_mean') and c != 'method']
    hm_data_norm = pd.DataFrame(MinMaxScaler().fit_transform(hm_data[hm_cols]), index=hm_data.index, columns=[c.replace('_mean','') for c in hm_cols])
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(hm_data_norm.T, annot=True, fmt='.2f', cmap='RdYlGn', annot_kws={'size': 8}, ax=ax)
    ax.set_title('Нормализованные метрики')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
except ImportError:
    print("seaborn/sklearn не установлены, пропускаем тепловую карту")

In [ ]:
print("="*70)
print("ОТЧЕТ ПО СРАВНЕНИЮ МЕТОДОВ")
print("="*70)
print(f"Дата: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Спектров: {len(TEST_SPECTRA)}, Методов: {len(SELECTED_METHODS)}")
print(f"Всего экспериментов: {len(results_df)}")
print("\nТОП-5 по R²:")
for i, row in ranking.head(5).iterrows():
    print(f"{i+1}. {row['method']:20s} R²={row['r2_score_mean']:.4f}")
print("\nУспешность (%):")
success = results_df.groupby('method')['success'].mean().sort_values(ascending=False)
for m, rate in success.items():
    print(f"{m:20s}: {rate*100:5.1f}%")
print("\nВремя (сек):")
for _, row in time_avg.sort_values('avg_time').iterrows():
    print(f"{row['method']:20s}: {row['avg_time']:8.3f}")
print("="*70)
print(f"Лучший по точности: {ranking.iloc[0]['method']}")
print(f"Самый быстрый: {time_avg.loc[time_avg['avg_time'].idxmin(), 'method']}")
print(f"Самый стабильный: {success.idxmax()}")
print("="*70)

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
results_df.to_csv(f'results_{ts}.csv', index=False)
summary_table.to_csv(f'summary_{ts}.csv', index=False)
print(f"Сохранено: results_{ts}.csv, summary_{ts}.csv")